In [ ]:
#-- R libraries
library(dplyr)
library(data.table)

#-- Copy from bucket back into your PD
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/PrescriptionData.csv"),
  "data/PrescriptionData.csv"
))

#-- Read into R
PrescriptionData <- read.csv("data/PrescriptionData.csv")

In [ ]:
#-- Arrange script data.frame
PrescriptionData <- PrescriptionData %>%
  mutate(drug_exposure_start_date = as.Date(drug_exposure_start_date)) %>%
  mutate(final_end_date = as.Date(final_end_date)) %>%
  group_by(person_id, drug) %>%
  arrange(person_id, drug, drug_exposure_start_date) %>%
  mutate(RowNum = row_number()) %>%
  mutate(PrescriptionEpisode = 1) %>%
  ungroup() %>%
  arrange(person_id, drug, drug_exposure_start_date) %>%
  data.table()

#-- Create copy of original data
PrescriptionData_Original <- PrescriptionData

In [ ]:
#-- Define a function to assign each prescription (P) to a prescription episode (PE)
MaxClearingWindow = 90
p2pe <- function(row_num, PrescriptionData) {
  current_prescription <- PrescriptionData %>%
    filter(RowNum == row_num)
  
  previous_prescription <-  PrescriptionData %>%
    filter(RowNum == row_num - 1)
  
  other_prescriptions <-  PrescriptionData %>% 
    filter(RowNum != row_num & RowNum != (row_num - 1))
  
  update_episode <- current_prescription %>%
    inner_join(previous_prescription, by = c("person_id", "drug"), suffix = c(".current", ".previous")) %>%
    mutate(
      PrescriptionEpisode.current = ifelse(
        as.numeric(difftime(drug_exposure_start_date.current, final_end_date.previous, units = "days")) > MaxClearingWindow,
        PrescriptionEpisode.previous + 1, 
        PrescriptionEpisode.previous
      )) %>%
    select(
      person_id, 
      drug,
      Category = Category.current,
      drug_exposure_start_date = drug_exposure_start_date.current,
      final_end_date = final_end_date.current,
      RowNum = RowNum.current,
      PrescriptionEpisode = PrescriptionEpisode.current
    )
  
  NewPrescriptionData <- rbind(previous_prescription, update_episode, other_prescriptions)
  return(NewPrescriptionData)
  
}

#-- Loop over each consecutive prescription to calculate prescription episodes
for (i in 2:max(PrescriptionData$RowNum)) {
  print(i/max(PrescriptionData$RowNum)*100)
  NewPrescriptionData <- p2pe(i, PrescriptionData)
  PrescriptionData <- as.data.frame(NewPrescriptionData) %>%
    arrange(person_id, drug, RowNum)
}


In [ ]:
# Check that all prescriptions got processed (YAY)
missing <- anti_join(PrescriptionData_Original, PrescriptionData, by = c("person_id", "drug", "drug_exposure_start_date"))
dim(missing)

In [ ]:
# Save locally first
write.csv(PrescriptionData, "data/AD_processed.csv", row.names = FALSE)

# Copy to workspace bucket
system(paste(
  "gsutil -u", Sys.getenv("GOOGLE_PROJECT"), "cp data/AD_processed.csv",
  file.path(Sys.getenv("WORKSPACE_BUCKET"), "results/AD_processed.csv")
))

